# 01 - Shipyard Environment Demo

This notebook demonstrates the HHI Ulsan shipyard simulation environment:
- Environment creation and configuration
- Observation and action spaces
- Running the expert scheduler
- Entity inspection (blocks, SPMTs, cranes, ships)
- Supply chain extension (suppliers, inventory, labor)
- Graph data for GNN encoding

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import yaml
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from simulation.shipyard_env import HHIShipyardEnv
from baselines.rule_based import RuleBasedScheduler

## 1. Load Configuration and Create Environment

In [ ]:
with open('../config/small_instance.yaml') as f:
    config = yaml.safe_load(f)

env = HHIShipyardEnv(config)
obs, info = env.reset(seed=42)

print(f"Ships: {env.n_ships}")
print(f"Blocks per ship: {env.n_blocks_per_ship}")
print(f"Total blocks: {env.n_ships * env.n_blocks_per_ship}")
print(f"SPMTs: {env.n_spmts}")
print(f"Goliath Cranes: {env.n_goliath_cranes}")
print(f"Dry Docks: {env.n_docks}")
print(f"Observation dim: {obs.shape}")
print(f"Action types: {env.n_action_types}")

## 2. Inspect Entities

In [ ]:
# Blocks
blocks = env.entities['blocks']
print(f"--- Blocks ({len(blocks)}) ---")
for b in blocks[:5]:
    print(f"  {b.id}: stage={b.current_stage.name}, status={b.status.name}, "
          f"weight={b.weight:.0f}t, due={b.due_date:.0f}h, ship={b.ship_id}")
print(f"  ...and {len(blocks)-5} more\n")

# SPMTs
spmts = env.entities['spmts']
print(f"--- SPMTs ({len(spmts)}) ---")
for s in spmts:
    print(f"  {s.id}: status={s.status.name}, capacity={s.capacity}t, "
          f"health=[H:{s.health_hydraulic:.0f} T:{s.health_tires:.0f} E:{s.health_engine:.0f}]")

# Goliath Cranes
cranes = env.entities['goliath_cranes']
print(f"\n--- Goliath Cranes ({len(cranes)}) ---")
for c in cranes:
    print(f"  {c.id}: dock={c.assigned_dock}, capacity={c.capacity_tons}t, "
          f"health=[H:{c.health_hoist:.0f} T:{c.health_trolley:.0f} G:{c.health_gantry:.0f}]")

# Ships
ships = env.entities['ships']
print(f"\n--- Ships ({len(ships)}) ---")
for s in ships:
    print(f"  {s.id}: status={s.status.name}, erected={s.blocks_erected}/{s.total_blocks}, "
          f"dock={s.assigned_dock}")

## 3. Action Masking

In [ ]:
mask = env.get_action_mask()
print("Action mask keys:", list(mask.keys()))
print(f"\nValid action types: {np.where(mask['action_type'])[0]}")
action_names = ['DISPATCH_SPMT', 'DISPATCH_CRANE', 'MAINTENANCE', 'HOLD']
for i, name in enumerate(action_names):
    if i < len(mask['action_type']):
        print(f"  {i}: {name} = {'VALID' if mask['action_type'][i] else 'masked'}")

print(f"\nTransport requests: {len(env.transport_requests)}")
print(f"Erection requests: {len(env.erection_requests)}")

## 4. Run Expert Scheduler for 500 Steps

In [ ]:
env.reset(seed=42)
expert = RuleBasedScheduler()

rewards = []
blocks_over_time = []
action_types_taken = []

for step in range(500):
    action = expert.decide(env)
    obs, reward, terminated, truncated, info = env.step(action)
    rewards.append(reward)
    blocks_over_time.append(env.metrics.get('blocks_erected', 0))
    action_types_taken.append(action['action_type'])
    if terminated or truncated:
        break

print(f"Steps completed: {len(rewards)}")
print(f"Total reward: {sum(rewards):.1f}")
print(f"Blocks erected: {env.metrics.get('blocks_erected', 0)}")
print(f"Ships delivered: {env.metrics.get('ships_delivered', 0)}")
print(f"Sim time: {env.sim_time:.0f} hours")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Cumulative blocks
axes[0].plot(blocks_over_time, color='steelblue', linewidth=2)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Blocks Erected')
axes[0].set_title('Production Progress')
axes[0].grid(True, alpha=0.3)

# Cumulative reward
axes[1].plot(np.cumsum(rewards), color='green', linewidth=2)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Cumulative Reward')
axes[1].set_title('Reward Accumulation')
axes[1].grid(True, alpha=0.3)

# Action type distribution
action_names_full = ['SPMT', 'Crane', 'Maint.', 'Hold']
counts = Counter(action_types_taken)
x = sorted(counts.keys())
labels = [action_names_full[i] if i < len(action_names_full) else f'Type {i}' for i in x]
axes[2].bar(labels, [counts[i] for i in x], color=['#4C72B0', '#DD8452', '#55A868', '#C44E52'][:len(x)])
axes[2].set_ylabel('Count')
axes[2].set_title('Action Distribution')

plt.tight_layout()
plt.show()

## 5. Block Stage Distribution

In [ ]:
stage_counts = Counter(b.current_stage.name for b in env.entities['blocks'])
stages = sorted(stage_counts.keys())
counts = [stage_counts[s] for s in stages]

fig, ax = plt.subplots(figsize=(12, 5))
colors = plt.cm.Set3(np.linspace(0, 1, len(stages)))
ax.barh(stages, counts, color=colors)
ax.set_xlabel('Number of Blocks')
ax.set_title(f'Block Stage Distribution (after {len(rewards)} steps)')
for i, v in enumerate(counts):
    ax.text(v + 0.5, i, str(v), va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 6. Equipment Health Tracking

In [ ]:
# Run a longer simulation tracking health
env.reset(seed=42)
expert = RuleBasedScheduler()

health_history = {s.id: [] for s in env.entities['spmts']}
crane_health_history = {c.id: [] for c in env.entities['goliath_cranes']}

for step in range(1000):
    for s in env.entities['spmts']:
        health_history[s.id].append(s.get_min_health())
    for c in env.entities['goliath_cranes']:
        crane_health_history[c.id].append(c.get_min_health())
    action = expert.decide(env)
    env.step(action)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, hist in health_history.items():
    axes[0].plot(hist, alpha=0.7, label=name)
axes[0].axhline(y=40, color='orange', linestyle='--', label='PM Threshold')
axes[0].axhline(y=20, color='red', linestyle='--', label='Failure')
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Min Health (%)')
axes[0].set_title('SPMT Health Over Time')
axes[0].legend(fontsize=7, ncol=2)
axes[0].grid(True, alpha=0.3)

for name, hist in crane_health_history.items():
    axes[1].plot(hist, alpha=0.7, linewidth=2, label=name)
axes[1].axhline(y=40, color='orange', linestyle='--', label='PM Threshold')
axes[1].axhline(y=20, color='red', linestyle='--', label='Failure')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Min Health (%)')
axes[1].set_title('Goliath Crane Health Over Time')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. GNN Graph Data

In [ ]:
graph_data = env.get_graph_data()
print("Node types:")
for nt in graph_data.node_types:
    x = graph_data[nt].x
    print(f"  {nt}: {x.shape[0]} nodes, {x.shape[1]} features")

print(f"\nEdge types ({len(graph_data.edge_types)}):")
for et in graph_data.edge_types:
    edge_index = graph_data[et].edge_index
    print(f"  {et[0]} --[{et[1]}]--> {et[2]}: {edge_index.shape[1]} edges")

## 8. Supply Chain Extension

In [ ]:
with open('../config/hhi_supply_chain.yaml') as f:
    sc_config = yaml.safe_load(f)

sc_env = HHIShipyardEnv(sc_config)
obs_sc, info_sc = sc_env.reset(seed=42)

print("=== Supply Chain Environment ===")
print(f"Observation dim: {obs_sc.shape} (base: {obs.shape})")
print(f"Action types: {sc_env.n_action_types} (base: 4)")
print(f"Suppliers: {sc_env.n_suppliers}")
print(f"Inventory types: {sc_env.n_inventory_nodes}")
print(f"Labor pools: {sc_env.n_labor_pools}")

# Inspect suppliers
print("\n--- Suppliers ---")
for s in sc_env.entities['suppliers']:
    print(f"  {s.id} ({s.name}): lead_time={s.lead_time_mean:.0f}h, "
          f"capacity={s.capacity_per_period:.0f}, price={s.price_per_unit:.2f}, "
          f"spec={s.specializations}")

# Inspect inventory
print("\n--- Inventory ---")
for inv in sc_env.entities['inventory']:
    print(f"  {inv.id}: type={inv.material_type.name}, qty={inv.quantity:.0f}/{inv.max_capacity:.0f}, "
          f"reorder_pt={inv.reorder_point:.0f}")

# Inspect labor pools
print("\n--- Labor Pools ---")
for lp in sc_env.entities['labor_pools']:
    print(f"  {lp.id}: skill={lp.skill_type.name}, workers={lp.available_workers}/{lp.total_workers}, "
          f"rate=${lp.hourly_rate:.0f}/hr")

In [ ]:
# Run expert on supply chain env
sc_env.reset(seed=42)
expert = RuleBasedScheduler()

sc_actions = []
inventory_levels = {inv.id: [] for inv in sc_env.entities['inventory']}
orders_over_time = []
cumulative_orders = 0

for step in range(500):
    for inv in sc_env.entities['inventory']:
        inventory_levels[inv.id].append(inv.quantity)
    action = expert.decide(sc_env)
    sc_actions.append(action['action_type'])
    obs, reward, terminated, truncated, info = sc_env.step(action)
    cumulative_orders = sc_env.metrics.get('orders_placed', 0)
    orders_over_time.append(cumulative_orders)
    if terminated or truncated:
        break

print(f"Blocks erected: {sc_env.metrics.get('blocks_erected', 0)}")
print(f"Orders placed: {sc_env.metrics.get('orders_placed', 0)}")
print(f"Deliveries: {sc_env.metrics.get('deliveries_received', 0)}")
print(f"Stockouts: {sc_env.metrics.get('stockout_events', 0)}")
print(f"Procurement cost: {sc_env.metrics.get('procurement_cost', 0):.0f}")
print(f"Holding cost: {sc_env.metrics.get('holding_cost', 0):.0f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Inventory levels over time
for name, levels in inventory_levels.items():
    axes[0].plot(levels, label=name, linewidth=1.5)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Quantity')
axes[0].set_title('Material Inventory Levels')
axes[0].legend(fontsize=7)
axes[0].grid(True, alpha=0.3)

# Cumulative orders
axes[1].plot(orders_over_time, color='darkorange', linewidth=2)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Cumulative Orders')
axes[1].set_title('Procurement Orders Over Time')
axes[1].grid(True, alpha=0.3)

# Action type distribution (supply chain)
action_names_sc = ['SPMT', 'Crane', 'Maint.', 'Hold', 'Order', 'Worker', 'Transfer']
counts_sc = Counter(sc_actions)
x_sc = sorted(counts_sc.keys())
labels_sc = [action_names_sc[i] if i < len(action_names_sc) else f'Type {i}' for i in x_sc]
colors_sc = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', '#937860', '#DA8BC3']
axes[2].bar(labels_sc, [counts_sc[i] for i in x_sc], color=[colors_sc[i] for i in x_sc])
axes[2].set_ylabel('Count')
axes[2].set_title('Action Distribution (Supply Chain)')

plt.tight_layout()
plt.show()

In [ ]:
# Supply chain graph data
sc_graph = sc_env.get_graph_data()
print("Supply Chain Graph:")
print(f"  Node types: {sc_graph.node_types}")
print(f"  Edge types: {len(sc_graph.edge_types)}")
for nt in sc_graph.node_types:
    x = sc_graph[nt].x
    print(f"  {nt}: {x.shape}")